In [1]:
ord("h")

104

In [2]:
ord("ក") # take only one input 



6016

In [3]:
[ord(x) for x in "សួស្តីបងប្រុស"]

[6047, 6077, 6047, 6098, 6031, 6072, 6036, 6020, 6036, 6098, 6042, 6075, 6047]

In [4]:
[ord(x) for x in "Hello brother"]

[72, 101, 108, 108, 111, 32, 98, 114, 111, 116, 104, 101, 114]

In [5]:
from pathlib import Path

CORPUS = Path("khmer_train_small.txt")
MAX_CHARS = None

text = CORPUS.read_text(encoding="utf-8")
if MAX_CHARS is not None:
    text = text[:MAX_CHARS]

print(f"{CORPUS.name}: {len(text):,} chars -> {len(text.encode('utf-8')):,} utf-8 bytes")

khmer_train_small.txt: 1,999,635 chars -> 5,794,024 utf-8 bytes


In [6]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

# a map is 

In [7]:
len(tokens)

5794024

In [8]:
def get_stats(ids):
    counts = {} 
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair,0) + 1
    return counts 


In [9]:
stats = get_stats(tokens)

top_pair = max(stats, key = stats.get)

print(top_pair)

count = stats[top_pair]

(225, 158)


In [10]:
raw_bytes = bytes([top_pair[0], top_pair[1]])
print(f"Top pair: {top_pair} -> {raw_bytes} (count: {count})")

Top pair: (225, 158) -> b'\xe1\x9e' (count: 1431687)


In [11]:
print(f"Decoded: {raw_bytes.decode('utf-8', errors='replace')}")

Decoded: �


In [12]:
def merge(ids, pair, idx):
    # ids:  the current list of token integers (e.g. [1, 2, 3, 1, 2])
    # pair: the tuple pair we want to replace (e.g. (1, 2))
    # idx:  the new token ID to replace the pair with (e.g. 256)
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2 
        else:
            new_ids.append(ids[i])
            i +=1 
    
    return new_ids

In [22]:
vocab_size = 4000
num_merges = vocab_size - 256
ids = list(tokens)

merges = {} # (int,int) 205,206 -> int 257

for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key= stats.get) # top pair of ids 
    idx = 256 + i
    ids = merge(ids, pair, idx) # one merge from the merge function 
    merges[pair] = idx

    

In [23]:
print(f"characters:        {len(text):,}")
print(f"bytes (tokens):    {len(tokens):,}")
print(f"after merges (ids):{len(ids):,}")
print(f"byte compression:  {len(tokens) / len(ids):.2f}X")
print(f"chars/token:       {len(text) / len(ids):.3f}")


characters:        1,999,635
bytes (tokens):    5,794,024
after merges (ids):617,235
byte compression:  9.39X
chars/token:       3.240


In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)} # ths is just looping all of the eng stuff

for (p0,p1),idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text 


In [25]:
def encode(ids):
    tokens = list(text.encode("utf-8"))
    



ក
   1 chars -> 1 tokens   round-trip True
ភ្នំពេញ
   7 chars -> 1 tokens   round-trip True
ព្រះរាជាណាចក្រកម្ពុជា
   21 chars -> 2 tokens   round-trip True
សួស្តី! Hello ១២៣ ៛
   19 chars -> 14 tokens   round-trip True


In [28]:
decode(encode("ព្រះរាជាណាចក្រកម្ពុជ"))


'ព្រះរាជាណាចក្រកម្ពុជ'

In [27]:
decode("")

KeyError: 'h'

In [29]:
(encode("ព្រះរាជាណាចក្រកម្ពុជ"))

[937, 1899, 294]

In [32]:
encode("937")
encode("57")

[53, 55]

In [ ]:
def encode(text):
    
    tokens = list(text.encod)